---

# ANALYSIS IDEAS

### HYPERPARAMETERS to change
- Forecast horizon length $H \in \{ 96, 192, 336, 720 \}$
- Number of periods $k \in \{ 1,2,3,4,5 \}$ or $k \in \{ 1,3,5 \}$ (We limit $k$ to small values because only dominant periodicities are meaningful)
- Input window length $T \in \{ 96, 192\}$, or simply keep 96 fixed

OPTIONAL (if time permits):
- aggregation method (mean vs weighted)
- kernel size in Conv2D

### RESULTS TO BE SHOWED
- <span style="color: green;">some plots where we show an actual sample and we compare the prediction with known future</span>
- <span style="color: green;">plot train and validation losses vs epochs</span>
- <span style="color: red;">plot comparison of the models (TimesNet vs baseline vs ...) using as structure: `Model | MSE | MAE | Time | Horizon `</span>

OPTIONAL (if time permits):
- period length analysis (see figure 10 of the paper): extract info on the periods identified by the FFT and make a final histogram (in the paper they found that the Electricity dataset contains many periods with length 12 and 24)
- show some 2D representations (after FFT) to make some smart comments
- test whether explicit period discovery improves prediction quality, i.e. if by using random periods instead of FFT we obtain similar reuslts, this would mean that learning also periods isn't so useful

P.S. All experiments are repeated three times to obtain an error

Things to remember from appendix:
* we select the d_model based on the input series dimension C by min{max{2^log Ce , dmin }, dmax }
* we use a MLP on temporal dimension to get the initialization of predicted future
* we also adopt the Series Stationarization from Non-stationary Transformer to eliminate the affect the distribution shift
* For the metrics, we adopt the mean square error (MSE) and mean absolute error (MAE) for long-term forecasting

---

### Which baseline to use

Best baseline: **1D CNN (TCN-like)**

Why this is the best choice:
* Easy to implement (no recurrence, no attention)
* Strong enough baseline (not trivial like linear regression)
* Direct comparison: 1D convolution vs 2D convolution on reshaped data

This is **exactly aligned with the paper motivation**: 1D models struggle with long-range / periodic patterns 

Other worst alternatives: LSTM, RNN, tranformers...

### Do I need full TimesNet?

No. Let’s separate essential vs optional parts.

ESSENTIAL:
- FFT-based period detection to select top-k frequencies
- 1D → 2D reshape. In this way: columns → intra-period, rows → inter-period 
- 2D processing (CNN). Paper uses inception, but I can replace with simple Conv2D. Paper itself says: any vision backbone can be used 
- Aggregation across periods. I can use softmax weights (like paper) or simple average

OPTIONAL (simplify / skip):
- Multi-layer TimesNet (stacking many blocks). I can use **1–2 blocks only**
- Complex embedding layers. I can use: Linear(input_dim → d_model)
- Inception module. I can replace it with: Conv2D → ReLU → Conv2D
- Advanced tricks (residuals, normalization): add only if I have time

The model becomes:

```
Input (T, C)
 → Linear embedding
 → FFT → top-k periods
 → reshape into k tensors (2D)
 → shared Conv2D
 → reshape back
 → aggregate
 → linear head → forecast
```


I can also write an ptional second “TimesNet variant”. Ideas:
- Remove FFT (no period detection). Use random or fixed periods, compare with learned periods. This directly tests: “Is explicit periodicity discovery useful?”

---

### OVERALL PLAN

#### STEP 0 — Setup
* Download Electricity dataset
* Load with pandas
* Normalize (VERY important)

#### STEP 1 — Data pipeline

Create sliding window dataset:
```python
X: (batch, T, C)
Y: (batch, H, C)
```

#### STEP 2 — Baseline model (1D CNN)

Simple architecture:

```python
Conv1D → ReLU → Conv1D → ReLU → Linear
```

Input:

```
(batch, T, C) → permute → (batch, C, T)
```

Output:

```
(batch, H, C)
```


#### STEP 3 — FFT + period extraction

Implement:

1. FFT over time dimension
2. compute amplitude
3. average over channels
4. take top-k frequencies, obtain periods

#### STEP 4 — Reshape to 2D

For each period:
* pad sequence
* reshape into:

```
(p_i, f_i, C)
```

#### STEP 5 — 2D CNN

Shared module:

```python
Conv2D → ReLU → Conv2D → pooling
```

#### STEP 6 — Back to 1D
* flatten 2D → 1D
* truncate padding

#### STEP 7 — Aggregate

With weighted sum or mean

#### STEP 8 — Forecast head

```python
Linear → predict H steps
```

#### STEP 9 — Training
* loss: MSE
* optimizer: Adam
* metrics: MSE, MAE

#### STEP 10 — Experiments

Minimum:
1. Baseline vs TimesNet-like → main result
2. Vary k → show periodicity effect
3. Vary horizon → short vs long forecasting

#### STEP 11 — Plots

* prediction vs ground truth
* loss curves
* performance vs k

# FROM VIDEO ABOUT PROJECT 2 - TIMESNET

### Learning from temporal data

* Time series appear in many real-world applications (e.g. energy, weather, finance, industrial monitoring)
* Forecasting and anomaly detection are key deep learning problems
* Real-world signals contain trends, seasonality, and multiple temporal patterns
* Capturing these variations is challenging


### Possible tasks: Forecasting and anomaly detection

Forecasting:
* Predict future values from past
* Useful in energy, weather, finance, 

Anomaly Detection
* Identify unusual or unexpected behavior in observations temporal signals
* Useful in monitoring, fault detection, demand forecasting cybersecurity


### TimesNet: from 1D signals to richer temporal representations 

* Real-world time series often exhibit multiple periodic patterns
* Directly modeling these variations in 1D can be challenging
* The signal is reshaped into 2D temporal representations
* This allows the model to capture patterns within and across periods 


### Core architectural idea 

* FFT is used to identify the most relevant periods
* The 1D signal is reshaped into a set of 2D temporal representations
* A shared TimesBlock / inception-style module processes these tensors
* Features from different periods are combined through adaptive aggregation
* The final representation is used for forecasting or anomal! detection

### Reference datasets 

Electricity 
* Real-world dataset for forecasting
* 370 electricity consumption series
* Measurements every 15 minutes
* Stored as TXT/CSV-style tabular data 

SMD (Server Machine Dataset) 
* Real-world dataset for anomaly detection
* Monitoring data from 28 server machines
* Collected over 5 weeks
* Multivariate time-series dataset with 38 dimensions 

Additional datasets: for forecasting ETT; for anomaly detection SWaT, SMAP, MSL, PSM.

### Possible project directions

Project 1 - Forecasting:
* Predict future values from past observations
* Reference dataset: Electricity
* Focus on periodic temporal structure
* Compare TimesNet-inspired ideas with simpler baselines
* Possible extensions:
    * study the effect of different forecasting horizons
    * compare 1D modeling with 2D periodicity-aware representations
    * simplify the TimesNet pipeline and evaluate what is lost
    * test whether explicit period discovery improves prediction quality
    * try additional datasets

Project 2 - Anomaly detection:
* Detect unusual temporal behavior in multivariate signals
* Reference dataset: SMD
* Focus on periodicity-aware anomaly detection
* Study whether structured temporal representations help identify anomalies
* Possible extensions:
    * compare detection performance with and without periodicity modeling
    * analyze which kinds of anomalies are easier or harder to detect
    * adapt only selected components of TimesNet to the anomaly task
    * study how temporal structure affects false positives and false negatives
    * try additional datasets

All of these are just ides and suggestions, you can also simplify the architecture, adapt the datasets, or focus on selected components of TimesNet rather then reproducing the full model. This proposals are only references and not to be considered as binding choices.

---

---

---

---

# MIXED IDEAS
- The real novelty is NOT the whole architecture, it is transforming 1D into multiple 2D tensors using FFT + periods, then process with 2D convs. So the project should focus on that.